# Recommendation System Project

## Contents

1. Problem Definition  
2. Dataset Description  
3. Data Preprocessing  
4. Baseline Model  
5. Embedding Model  
6. Neural Two-Tower Model  
7. Feature-Enriched Two-Tower  
8. Evaluation  
9. FAISS-Based Retrieval  
10. Ranking Model (Second Stage)  
11. Example Recommendations  
12. Error Analysis  
13. Diversity Analysis  
14. Production Pipeline  
15. Limitations  
16. Future Work  

In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip ml-100k.zip

--2026-03-24 21:27:16--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip.1’

ml-100k.zip.1       100%[===================>]   4.70M  9.20MB/s    in 0.5s    

2026-03-24 21:27:17 (9.20 MB/s) - ‘ml-100k.zip.1’ saved [4924029/4924029]

Archive:  ml-100k.zip
replace ml-100k/allbut.pl? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [2]:
import pandas as pd

df = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

df["interaction"] = 1
df = df[["user_id", "item_id", "interaction"]]

df.head()

,user_id,item_id,interaction
0,196,242,1
1,186,302,1
2,22,377,1
3,244,51,1
4,166,346,1


In [3]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [4]:
print(train_df.head())
print(test_df.head())

       user_id  item_id  interaction
75220      807     1411            1
48955      474      659            1
44966      463      268            1
13568      139      286            1
92727      621      751            1
       user_id  item_id  interaction
75721      877      381            1
80184      815      602            1
19864       94      431            1
76699      416      875            1
92991      500      182            1


In [5]:
popularity = train_df.groupby("item_id").size().reset_index(name="count")

popularity = popularity.sort_values(by="count", ascending=False)

popularity.head()

,item_id,count
49,50,451
99,100,408
180,181,406
257,258,397
287,288,390


In [6]:
TOP_K = 10
top_k_items = popularity["item_id"].head(TOP_K).tolist()

print(top_k_items)

[50, 100, 181, 258, 288, 286, 294, 300, 1, 121]


In [7]:
def recall_at_k(test_df, top_k_items):
    hits = 0
    total = 0

    for user in test_df["user_id"].unique():
        user_items = test_df[test_df["user_id"] == user]["item_id"].tolist()

        for item in user_items:
            if item in top_k_items:
                hits += 1
        total += len(user_items)

    return hits / total

In [8]:
recall = recall_at_k(test_df, top_k_items)
print("Recall@10:", recall)

Recall@10: 0.04985


In [9]:
import pandas as pd

ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

ratings["interaction"] = 1
ratings = ratings[["user_id", "item_id", "interaction"]]

ratings.head()

,user_id,item_id,interaction
0,196,242,1
1,186,302,1
2,22,377,1
3,244,51,1
4,166,346,1


In [10]:
unique_users = ratings["user_id"].unique()
unique_items = ratings["item_id"].unique()

user2idx = {user: idx for idx, user in enumerate(unique_users)}
item2idx = {item: idx for idx, item in enumerate(unique_items)}

ratings["user_idx"] = ratings["user_id"].map(user2idx)
ratings["item_idx"] = ratings["item_id"].map(item2idx)

ratings.head()

,user_id,item_id,interaction,user_idx,item_idx
0,196,242,1,0,0
1,186,302,1,1,1
2,22,377,1,2,2
3,244,51,1,3,3
4,166,346,1,4,4


In [11]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    ratings[["user_id", "item_id", "user_idx", "item_idx", "interaction"]],
    test_size=0.2,
    random_state=42
)

In [12]:
import numpy as np

all_item_ids = set(ratings["item_id"].unique())

user_positive_items = ratings.groupby("user_id")["item_id"].apply(set).to_dict()

def generate_negative_samples(df, num_negatives=4):
    negative_rows = []

    for _, row in df.iterrows():
        user_id = row["user_id"]
        positive_items = user_positive_items[user_id]

        for _ in range(num_negatives):
            negative_item = np.random.choice(list(all_item_ids))
            while negative_item in positive_items:
                negative_item = np.random.choice(list(all_item_ids))

            negative_rows.append({
                "user_id": user_id,
                "item_id": negative_item,
                "user_idx": user2idx[user_id],
                "item_idx": item2idx[negative_item],
                "interaction": 0
            })

    negative_df = pd.DataFrame(negative_rows)
    return negative_df

In [13]:
train_neg_df = generate_negative_samples(train_df, num_negatives=1)

train_final = pd.concat([train_df, train_neg_df], ignore_index=True)
train_final = train_final.sample(frac=1, random_state=42).reset_index(drop=True)

train_final.head()

,user_id,item_id,user_idx,item_idx,interaction
0,256,1379,117,1327,0
1,279,1000,75,916,1
2,329,11,319,229,1
3,429,162,421,540,1
4,339,152,333,503,0


In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [15]:
class RecDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_idx"].values, dtype=torch.long)
        self.labels = torch.tensor(df["interaction"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

In [16]:
class TwoTowerMini(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

    def forward(self, user_ids, item_ids):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)

        scores = (user_vec * item_vec).sum(dim=1)
        return scores

In [17]:
train_dataset = RecDataset(train_final)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

num_users = len(user2idx)
num_items = len(item2idx)

model = TwoTowerMini(num_users, num_items, embedding_dim=32)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [18]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for users, items, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(users, items)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

Epoch 1/10, Loss: 1.9402
Epoch 2/10, Loss: 1.0419
Epoch 3/10, Loss: 0.6220
Epoch 4/10, Loss: 0.4395
Epoch 5/10, Loss: 0.3623
Epoch 6/10, Loss: 0.3150
Epoch 7/10, Loss: 0.2795
Epoch 8/10, Loss: 0.2511
Epoch 9/10, Loss: 0.2262
Epoch 10/10, Loss: 0.2044


In [19]:
def recall_at_10_model(model, train_df, test_df, all_items, user2idx, item2idx, k=10):
    model.eval()
    recalls = []

    train_user_items = train_df.groupby("user_id")["item_id"].apply(set).to_dict()
    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    with torch.no_grad():
        for user_id in test_user_items:
            if user_id not in user2idx:
                continue

            seen_items = train_user_items.get(user_id, set())
            true_items = test_user_items[user_id]

            candidate_items = [item for item in all_items if item not in seen_items]
            candidate_item_idxs = [item2idx[item] for item in candidate_items]

            user_idx = user2idx[user_id]
            user_tensor = torch.tensor([user_idx] * len(candidate_item_idxs), dtype=torch.long)
            item_tensor = torch.tensor(candidate_item_idxs, dtype=torch.long)

            scores = model(user_tensor, item_tensor)
            top_k_indices = torch.topk(scores, k).indices.numpy()
            recommended_items = [candidate_items[i] for i in top_k_indices]

            hits = len(set(recommended_items) & true_items)
            recall = hits / len(true_items) if len(true_items) > 0 else 0
            recalls.append(recall)

    return np.mean(recalls)

In [20]:
all_items_list = list(ratings["item_id"].unique())

recall_model = recall_at_10_model(
    model,
    train_df,
    test_df,
    all_items_list,
    user2idx,
    item2idx,
    k=10
)

print("Embedding Model Recall@10:", recall_model)

Embedding Model Recall@10: 0.05032329118431225


## Sonuçların Özeti

Bu çalışmada üç farklı yaklaşım karşılaştırılmıştır:

1. Popularity-based baseline
2. Temel embedding modeli
3. İyileştirilmiş embedding modeli

## Performans Karşılaştırması

- Popularity Baseline: **0.04985**
- Temel Embedding Modeli: **0.03323**
- İyileştirilmiş Embedding Modeli: **0.05032**

## Yorum

Popularity baseline, kişiselleştirme içermemesine rağmen güçlü bir başlangıç performansı sunmuştur. İlk embedding modelinin baseline'ın altında kalması, recommendation problemlerinde yalnızca basit latent temsil öğrenmenin yeterli olmayabileceğini göstermiştir.

Uygulanan iyileştirmeler sonrasında embedding modelinin performance artmış ve Recall@10 metriğinde baseline sonucu hafifçe aşılmıştır. Bu durum, negatif örnekleme ve hiperparametre optimizasyonunun model başarısında kritik rol oynadığını göstermektedir.

## Summary of Results

In this study, three different approaches were compared:

1. Popularity-based baseline  
2. Basic embedding model  
3. Improved embedding model  

## Performance Comparison

- Popularity Baseline: **0.04985**  
- Basic Embedding Model: **0.03323**  
- Improved Embedding Model: **0.05032**  

## Discussion

The popularity baseline, despite not incorporating any personalization, provided a strong initial performance. The fact that the basic embedding model performed below the baseline indicates that simple latent representation learning alone may not be sufficient for recommendation tasks.

After applying improvements such as enhanced negative sampling and hyperparameter tuning, the embedding model's performance increased and slightly surpassed the baseline in terms of Recall@10. This result highlights the critical role of negative sampling strategies and hyperparameter optimization in improving model performance.

## Model Sonuçlarının Değerlendirilmesi

Bu çalışmada recommendation performansını değerlendirmek amacıyla üç farklı yaklaşım karşılaştırılmıştır: popularity baseline, temel embedding modeli ve iyileştirilmiş embedding modeli.

## Performans Tablosu

| Model | Recall@10 |
|------|-----------:|
| Popularity Baseline | 0.04985 |
| İlk Embedding Model | 0.03323 |
| İyileştirilmiş Embedding Model | 0.05032 |

## Yorum

Popularity baseline modeli, tüm kullanıcılara en popüler item'ları önererek güçlü bir referans performans sunmuştur. Buna karşılık, ilk embedding modeli kullanıcı ve item latent temsillerini öğrenmesine rağmen baseline performansının gerisinde kalmıştır. Bu durum, başlangıç aşamasında kullanılan negatif örnekleme stratejisinin ve hiperparametre seçimlerinin model performansını sınırladığını göstermektedir.

Daha sonra negatif örnek sayısının artırılması, embedding boyutunun büyütülmesi, öğrenme oranının düşürülmesi ve eğitim süresinin uzatılması gibi iyileştirmeler uygulanmıştır. Bu geliştirmeler sonucunda embedding modelinin Recall@10 skoru 0.03323 seviyesinden 0.05032 seviyesine yükselmiş ve popularity baseline hafif de olsa aşılmıştır.

Elde edilen sonuçlar, recommendation problemlerinde embedding tabanlı kişiselleştirilmiş modellerin uygun eğitim stratejileri ile baseline yöntemlerin üzerine çıkabileceğini göstermektedir.

## Sonraki Adımlar

Bir sonraki aşamada gerçek Two-Tower mimarisi, ek kullanıcı-item özellikleri ve daha gelişmiş değerlendirme metrikleri kullanılarak model daha da geliştirilebilir.

## Evaluation of Model Results

In this study, three different approaches were compared to evaluate recommendation performance: a popularity-based baseline, a basic embedding model, and an improved embedding model.

## Performance Table

| Model | Recall@10 |
|------|-----------:|
| Popularity Baseline | 0.04985 |
| Initial Embedding Model | 0.03323 |
| Improved Embedding Model | 0.05032 |

## Discussion

The popularity baseline model provided a strong reference performance by recommending the most popular items to all users, despite not incorporating any personalization. In contrast, the initial embedding model, although capable of learning latent representations of users and items, performed below the baseline. This indicates that the initial negative sampling strategy and hyperparameter choices were not sufficient to fully capture user-item relationships.

Subsequently, several improvements were applied, including increasing the number of negative samples, enlarging the embedding dimension, reducing the learning rate, and extending the training duration. As a result of these enhancements, the embedding model's Recall@10 score improved from 0.03323 to 0.05032, slightly surpassing the popularity baseline.

These findings demonstrate that embedding-based personalized recommendation models can outperform baseline methods when supported by appropriate training strategies and hyperparameter tuning.

## Next Steps

In the next stage, the model can be further improved by implementing a full Two-Tower architecture, incorporating additional user and item features, and utilizing more advanced evaluation metrics.

In [21]:
class TwoTowerNN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()

        # Embedding
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        # User tower
        self.user_fc = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        # Item tower
        self.item_fc = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

    def forward(self, user_ids, item_ids):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)

        user_vec = self.user_fc(user_vec)
        item_vec = self.item_fc(item_vec)

        scores = (user_vec * item_vec).sum(dim=1)
        return scores

In [22]:
model = TwoTowerNN(num_users, num_items, embedding_dim=64)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [23]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for users, items, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(users, items)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

Epoch 1, Loss: 0.5756
Epoch 2, Loss: 0.4798
Epoch 3, Loss: 0.4664
Epoch 4, Loss: 0.4586
Epoch 5, Loss: 0.4509
Epoch 6, Loss: 0.4415
Epoch 7, Loss: 0.4297
Epoch 8, Loss: 0.4139
Epoch 9, Loss: 0.3965
Epoch 10, Loss: 0.3798


In [25]:
def recall_at_10_model(model, train_df, test_df, all_items, user2idx, item2idx, k=10):
    model.eval()
    recalls = []

    train_user_items = train_df.groupby("user_id")["item_id"].apply(set).to_dict()
    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    with torch.no_grad():
        for user_id in test_user_items:
            if user_id not in user2idx:
                continue

            seen_items = train_user_items.get(user_id, set())
            true_items = test_user_items[user_id]

            candidate_items = [item for item in all_items if item not in seen_items]
            if len(candidate_items) < k:
                continue

            candidate_item_idxs = [item2idx[item] for item in candidate_items]

            user_idx = user2idx[user_id]
            user_tensor = torch.tensor([user_idx] * len(candidate_item_idxs), dtype=torch.long)
            item_tensor = torch.tensor(candidate_item_idxs, dtype=torch.long)

            scores = model(user_tensor, item_tensor)
            top_k_indices = torch.topk(scores, k).indices.numpy()
            recommended_items = [candidate_items[i] for i in top_k_indices]

            hits = len(set(recommended_items) & true_items)
            recall = hits / len(true_items) if len(true_items) > 0 else 0
            recalls.append(recall)

    return np.mean(recalls)

In [26]:
all_items_list = list(ratings["item_id"].unique())

recall_model = recall_at_10_model(
    model,
    train_df,
    test_df,
    all_items_list,
    user2idx,
    item2idx,
    k=10
)

print("Neural Two-Tower Recall@10:", recall_model)

Neural Two-Tower Recall@10: 0.10921556855534091


In [56]:
import pandas as pd

item_columns = [
    "movie_id", "movie_title", "release_date", "video_release_date", "IMDb_URL",
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
    "Romance", "Sci-Fi", "Thriller", "War", "Western"
]

items = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    names=item_columns
)

items.head()

,movie_id,movie_title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [57]:
genre_columns = [
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
    "Romance", "Sci-Fi", "Thriller", "War", "Western"
]

item_features = items[["movie_id"] + genre_columns].copy()
item_features.head()

,movie_id,unknown,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,4,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0


In [58]:
ratings_with_features = ratings.merge(
    item_features,
    left_on="item_id",
    right_on="movie_id",
    how="left"
)

ratings_with_features.head()

,user_id,item_id,interaction,user_idx,item_idx,movie_id,unknown,Action,Adventure,Animation,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,196,242,1,0,0,242,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,186,302,1,1,1,302,0,0,0,0,...,0,1,0,0,1,0,0,1,0,0
2,22,377,1,2,2,377,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,244,51,1,3,3,51,0,0,0,0,...,0,0,0,0,0,1,0,0,1,1
4,166,346,1,4,4,346,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [59]:
ratings_with_features = ratings_with_features.drop(columns=["movie_id"])

In [33]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    ratings_with_features,
    test_size=0.2,
    random_state=42
)

In [60]:
all_item_ids = set(ratings_with_features["item_id"].unique())
user_positive_items = ratings_with_features.groupby("user_id")["item_id"].apply(set).to_dict()

item_feature_lookup = item_features.set_index("movie_id").to_dict(orient="index")

def generate_negative_samples_with_features(df, num_negatives=3):
    negative_rows = []

    for _, row in df.iterrows():
        user_id = row["user_id"]
        positive_items = user_positive_items[user_id]

        for _ in range(num_negatives):
            negative_item = np.random.choice(list(all_item_ids))
            while negative_item in positive_items:
                negative_item = np.random.choice(list(all_item_ids))

            feature_dict = item_feature_lookup[negative_item]

            negative_row = {
                "user_id": user_id,
                "item_id": negative_item,
                "interaction": 0,
                "user_idx": user2idx[user_id],
                "item_idx": item2idx[negative_item],
            }

            for col in genre_columns:
                negative_row[col] = feature_dict[col]

            negative_rows.append(negative_row)

    return pd.DataFrame(negative_rows)

In [35]:
base_columns = ["user_id", "item_id", "interaction", "user_idx", "item_idx"] + genre_columns
train_df = train_df[base_columns]
test_df = test_df[base_columns]

In [36]:
train_neg_df = generate_negative_samples_with_features(train_df, num_negatives=3)

train_final = pd.concat([train_df, train_neg_df], ignore_index=True)
train_final = train_final.sample(frac=1, random_state=42).reset_index(drop=True)

train_final.head()

,user_id,item_id,interaction,user_idx,item_idx,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,715,560,0,706,841,0,0,1,0,1,...,1,0,0,0,0,1,1,0,0,0
1,585,108,0,580,1006,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,291,1123,0,18,1160,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,184,633,0,162,1028,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,417,1146,0,413,1498,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
import torch
from torch.utils.data import Dataset, DataLoader

class RecDatasetWithFeatures(Dataset):
    def __init__(self, df, genre_columns):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_idx"].values, dtype=torch.long)
        self.genres = torch.tensor(df[genre_columns].values, dtype=torch.float32)
        self.labels = torch.tensor(df["interaction"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.genres[idx], self.labels[idx]

In [39]:
import torch.nn as nn

class TwoTowerWithFeatures(nn.Module):
    def __init__(self, num_users, num_items, num_genres, embedding_dim=64):
        super().__init__()

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        self.user_tower = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.item_tower = nn.Sequential(
            nn.Linear(embedding_dim + num_genres, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

    def forward(self, user_ids, item_ids, genre_feats):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)

        user_out = self.user_tower(user_vec)

        item_input = torch.cat([item_vec, genre_feats], dim=1)
        item_out = self.item_tower(item_input)

        scores = (user_out * item_out).sum(dim=1)
        return scores

In [40]:
train_dataset = RecDatasetWithFeatures(train_final, genre_columns)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

num_users = len(user2idx)
num_items = len(item2idx)
num_genres = len(genre_columns)

model = TwoTowerWithFeatures(
    num_users=num_users,
    num_items=num_items,
    num_genres=num_genres,
    embedding_dim=64
)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [41]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for users, items, genres, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(users, items, genres)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

Epoch 1/10 - Loss: 0.4601
Epoch 2/10 - Loss: 0.3987
Epoch 3/10 - Loss: 0.3913
Epoch 4/10 - Loss: 0.3839
Epoch 5/10 - Loss: 0.3712
Epoch 6/10 - Loss: 0.3544
Epoch 7/10 - Loss: 0.3368
Epoch 8/10 - Loss: 0.3241
Epoch 9/10 - Loss: 0.3157
Epoch 10/10 - Loss: 0.3091


In [42]:
def recall_at_10_model_with_features(model, train_df, test_df, all_items, user2idx, item2idx, item_features, genre_columns, k=10):
    model.eval()
    recalls = []

    train_user_items = train_df.groupby("user_id")["item_id"].apply(set).to_dict()
    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    item_features_lookup = item_features.set_index("movie_id")

    with torch.no_grad():
        for user_id in test_user_items:
            if user_id not in user2idx:
                continue

            seen_items = train_user_items.get(user_id, set())
            true_items = test_user_items[user_id]

            candidate_items = [item for item in all_items if item not in seen_items]
            if len(candidate_items) < k:
                continue

            candidate_item_idxs = [item2idx[item] for item in candidate_items]
            candidate_genres = item_features_lookup.loc[candidate_items, genre_columns].values

            user_idx = user2idx[user_id]
            user_tensor = torch.tensor([user_idx] * len(candidate_items), dtype=torch.long)
            item_tensor = torch.tensor(candidate_item_idxs, dtype=torch.long)
            genre_tensor = torch.tensor(candidate_genres, dtype=torch.float32)

            scores = model(user_tensor, item_tensor, genre_tensor)
            top_k_indices = torch.topk(scores, k).indices.numpy()
            recommended_items = [candidate_items[i] for i in top_k_indices]

            hits = len(set(recommended_items) & true_items)
            recall = hits / len(true_items) if len(true_items) > 0 else 0
            recalls.append(recall)

    return np.mean(recalls)

In [43]:
all_items_list = list(ratings_with_features["item_id"].unique())

recall_with_features = recall_at_10_model_with_features(
    model=model,
    train_df=train_df,
    test_df=test_df,
    all_items=all_items_list,
    user2idx=user2idx,
    item2idx=item2idx,
    item_features=item_features,
    genre_columns=genre_columns,
    k=10
)

print("Two-Tower with Genre Features Recall@10:", recall_with_features)

Two-Tower with Genre Features Recall@10: 0.15779551541808337


Incorporating item-side metadata (genre features) significantly improved model performance, increasing Recall@10 from 0.109 to 0.158. This demonstrates that combining learned embeddings with explicit item features enables the model to capture both behavioral patterns and semantic information, leading to better generalization.

In [44]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.3 MB/s eta 0:00:00


In [45]:
model.eval()

item_embeddings = []

with torch.no_grad():
    for item_id in item2idx:
        idx = item2idx[item_id]

        item_tensor = torch.tensor([idx], dtype=torch.long)

        # genre al
        genre = item_features[item_features["movie_id"] == item_id][genre_columns].values
        genre_tensor = torch.tensor(genre, dtype=torch.float32)

        item_vec = model.item_embedding(item_tensor)
        item_vec = torch.cat([item_vec, genre_tensor], dim=1)
        item_vec = model.item_tower(item_vec)

        item_embeddings.append(item_vec.numpy()[0])

In [46]:
import numpy as np

item_embeddings = np.array(item_embeddings).astype("float32")

In [47]:
import faiss

dimension = item_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(item_embeddings)

print("Index hazır:", index.ntotal)

Index hazır: 1682


In [48]:
def get_user_embedding(user_id):
    model.eval()

    with torch.no_grad():
        user_idx = user2idx[user_id]
        user_tensor = torch.tensor([user_idx], dtype=torch.long)

        user_vec = model.user_embedding(user_tensor)
        user_vec = model.user_tower(user_vec)

    return user_vec.numpy()

In [49]:
def recommend_faiss(user_id, top_k=10):
    user_vec = get_user_embedding(user_id)

    distances, indices = index.search(user_vec, top_k)

    recommended_items = []
    item_ids = list(item2idx.keys())

    for idx in indices[0]:
        recommended_items.append(item_ids[idx])

    return recommended_items

In [50]:
user_id = 1

recommendations = recommend_faiss(user_id, top_k=10)

print("Önerilen item'lar:", recommendations)

Önerilen item'lar: [np.int64(100), np.int64(258), np.int64(172), np.int64(181), np.int64(222), np.int64(56), np.int64(237), np.int64(4), np.int64(7), np.int64(64)]


In [51]:
def recommend_faiss(user_id, top_k=10):
    user_vec = get_user_embedding(user_id)

    distances, indices = index.search(user_vec, top_k)

    recommended_items = []
    item_ids = list(item2idx.keys())

    for idx in indices[0]:
        recommended_items.append(int(item_ids[idx]))

    return recommended_items

In [52]:
user_id = 1
recommendations = recommend_faiss(user_id, top_k=10)
print("Recommended item IDs:", recommendations)

Recommended item IDs: [100, 258, 172, 181, 222, 56, 237, 4, 7, 64]


In [53]:
train_user_items = train_df.groupby("user_id")["item_id"].apply(set).to_dict()

def recommend_faiss_filtered(user_id, top_k=10, search_k=50):
    user_vec = get_user_embedding(user_id)

    distances, indices = index.search(user_vec, search_k)

    seen_items = train_user_items.get(user_id, set())
    item_ids = list(item2idx.keys())

    recommended_items = []

    for idx in indices[0]:
        item_id = int(item_ids[idx])

        if item_id not in seen_items:
            recommended_items.append(item_id)

        if len(recommended_items) == top_k:
            break

    return recommended_items

In [54]:
user_id = 1
filtered_recommendations = recommend_faiss_filtered(user_id, top_k=10, search_k=50)

print("Filtered recommendations:", filtered_recommendations)

Filtered recommendations: [100, 258, 181, 237, 4, 64, 273, 405, 748, 204]


In [61]:
movie_lookup = items.set_index("movie_id")["movie_title"].to_dict()

def show_recommendations_with_titles(item_ids):
    for item_id in item_ids:
        print(f"{item_id}: {movie_lookup.get(item_id, 'Unknown Title')}")

In [62]:
show_recommendations_with_titles(filtered_recommendations)

100: Fargo (1996)
258: Contact (1997)
181: Return of the Jedi (1983)
237: Jerry Maguire (1996)
4: Get Shorty (1995)
64: Shawshank Redemption, The (1994)
273: Heat (1995)
405: Mission: Impossible (1996)
748: Saint, The (1997)
204: Back to the Future (1985)


## Example Recommendations

After training the Two-Tower model with genre features and indexing item representations with FAISS, the system was able to generate realistic movie recommendations for individual users. The retrieved items included well-known and semantically relevant movies such as *Fargo (1996)*, *Contact (1997)*, *Return of the Jedi (1983)*, and *The Shawshank Redemption (1994)*.

This suggests that the model is not only learning user-item interaction patterns, but is also benefiting from genre-based item metadata to produce more meaningful recommendations.
*italik metin*

In [63]:
train_user_items = train_df.groupby("user_id")["item_id"].apply(set).to_dict()

def recommend_faiss_filtered(user_id, top_k=10, search_k=100):
    user_vec = get_user_embedding(user_id)

    distances, indices = index.search(user_vec, search_k)

    seen_items = train_user_items.get(user_id, set())
    item_ids = list(item2idx.keys())

    recommended_items = []

    for idx in indices[0]:
        item_id = int(item_ids[idx])

        if item_id not in seen_items:
            recommended_items.append(item_id)

        if len(recommended_items) == top_k:
            break

    return recommended_items

In [64]:
def recall_at_10_faiss(test_df, top_k=10, search_k=100):
    recalls = []

    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    for user_id, true_items in test_user_items.items():
        if user_id not in user2idx:
            continue

        recommended_items = recommend_faiss_filtered(
            user_id=user_id,
            top_k=top_k,
            search_k=search_k
        )

        hits = len(set(recommended_items) & true_items)
        recall = hits / len(true_items) if len(true_items) > 0 else 0
        recalls.append(recall)

    return np.mean(recalls)

In [65]:
faiss_recall = recall_at_10_faiss(test_df, top_k=10, search_k=100)
print("FAISS Recall@10:", faiss_recall)

FAISS Recall@10: 0.14118014741921503


The FAISS-based retrieval achieved a Recall@10 of 0.14118, which is slightly lower than the full scoring approach (0.15779). This indicates a small trade-off between accuracy and efficiency.

However, the result demonstrates that the learned embedding space is well-structured, allowing FAISS to retrieve highly relevant items with minimal loss in recommendation quality, while significantly improving inference speed and scalability.


In [66]:
def precision_at_10_faiss(test_df, top_k=10, search_k=100):
    precisions = []

    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    for user_id, true_items in test_user_items.items():
        if user_id not in user2idx:
            continue

        recommended_items = recommend_faiss_filtered(
            user_id=user_id,
            top_k=top_k,
            search_k=search_k
        )

        hits = len(set(recommended_items) & true_items)
        precision = hits / top_k
        precisions.append(precision)

    return np.mean(precisions)


def hit_rate_at_10_faiss(test_df, top_k=10, search_k=100):
    hits_list = []

    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    for user_id, true_items in test_user_items.items():
        if user_id not in user2idx:
            continue

        recommended_items = recommend_faiss_filtered(
            user_id=user_id,
            top_k=top_k,
            search_k=search_k
        )

        hit = 1 if len(set(recommended_items) & true_items) > 0 else 0
        hits_list.append(hit)

    return np.mean(hits_list)

In [67]:
faiss_precision = precision_at_10_faiss(test_df, top_k=10, search_k=100)
faiss_hit_rate = hit_rate_at_10_faiss(test_df, top_k=10, search_k=100)

print("FAISS Precision@10:", faiss_precision)
print("FAISS Hit Rate@10:", faiss_hit_rate)

FAISS Precision@10: 0.24553191489361703
FAISS Hit Rate@10: 0.8106382978723404


The FAISS-based recommendation system achieved a Recall@10 of 0.141, Precision@10 of 0.245, and Hit Rate@10 of 0.811. These results indicate that the system is able to retrieve a substantial portion of relevant items while maintaining a reasonable level of recommendation accuracy.

In particular, the high Hit Rate@10 suggests that the model is capable of providing at least one relevant recommendation for the majority of users, which is a desirable property in real-world recommendation systems.

In [68]:
def user_level_recall_faiss(test_df, top_k=10, search_k=100):
    user_recalls = {}

    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    for user_id, true_items in test_user_items.items():
        if user_id not in user2idx:
            continue

        recommended_items = recommend_faiss_filtered(
            user_id=user_id,
            top_k=top_k,
            search_k=search_k
        )

        hits = len(set(recommended_items) & true_items)
        recall = hits / len(true_items) if len(true_items) > 0 else 0

        user_recalls[user_id] = recall

    return user_recalls

## Error Analysis

This section analyzes model performance at the user level in order to better understand where the system performs well and where it struggles. By examining best and worst performing users, we can identify patterns related to user behavior, data sparsity, and model bias.

In [69]:
user_recalls = user_level_recall_faiss(test_df)

# en iyi 5 kullanıcı
best_users = sorted(user_recalls.items(), key=lambda x: x[1], reverse=True)[:5]

# en kötü 5 kullanıcı
worst_users = sorted(user_recalls.items(), key=lambda x: x[1])[:5]

print("Best users:", best_users)
print("Worst users:", worst_users)

Best users: [(66, 0.75), (926, 0.75), (657, 0.6666666666666666), (888, 0.6666666666666666), (589, 0.6363636363636364)]
Worst users: [(4, 0.0), (5, 0.0), (8, 0.0), (9, 0.0), (12, 0.0)]


The results show that the model performs significantly better for certain users while struggling for others.

In [70]:
user_id = best_users[0][0]

In [71]:
true_items = test_df[test_df["user_id"] == user_id]["item_id"].tolist()
print("Gerçek:", true_items)

Gerçek: [121, 258, 15, 295]


In [72]:
recs = recommend_faiss_filtered(user_id)
print("Öneri:", recs)

Öneri: [121, 258, 100, 748, 313, 678, 125, 268, 15, 271]


In [73]:
print("\nGerçek filmler:")
show_recommendations_with_titles(true_items)

print("\nÖnerilen filmler:")
show_recommendations_with_titles(recs)


Gerçek filmler:
121: Independence Day (ID4) (1996)
258: Contact (1997)
15: Mr. Holland's Opus (1995)
295: Breakdown (1997)

Önerilen filmler:
121: Independence Day (ID4) (1996)
258: Contact (1997)
100: Fargo (1996)
748: Saint, The (1997)
313: Titanic (1997)
678: Volcano (1997)
125: Phenomenon (1996)
268: Chasing Amy (1997)
15: Mr. Holland's Opus (1995)
271: Starship Troopers (1997)


In [74]:
user_id = worst_users[0][0]

In [75]:
true_items = test_df[test_df["user_id"] == user_id]["item_id"].tolist()
print("Gerçek:", true_items)

Gerçek: [358, 50, 356, 271]


In [76]:
recs = recommend_faiss_filtered(user_id)
print("Öneri:", recs)

Öneri: [286, 269, 313, 127, 322, 272, 289, 268, 690, 748]


In [77]:
print("\nGerçek filmler:")
show_recommendations_with_titles(true_items)

print("\nÖnerilen filmler:")
show_recommendations_with_titles(recs)


Gerçek filmler:
358: Spawn (1997)
50: Star Wars (1977)
356: Client, The (1994)
271: Starship Troopers (1997)

Önerilen filmler:
286: English Patient, The (1996)
269: Full Monty, The (1997)
313: Titanic (1997)
127: Godfather, The (1972)
322: Murder at 1600 (1997)
272: Good Will Hunting (1997)
289: Evita (1996)
268: Chasing Amy (1997)
690: Seven Years in Tibet (1997)
748: Saint, The (1997)


In [78]:
popularity = train_df.groupby("item_id").size().to_dict()

def avg_popularity(items):
    return np.mean([popularity.get(i, 0) for i in items])

In [79]:
user_id = best_users[0][0]

recs = recommend_faiss_filtered(user_id)

print("Avg popularity of recommendations:", avg_popularity(recs))

Avg popularity of recommendations: 266.2


In [80]:
def get_genres(item_id):
    row = item_features[item_features["movie_id"] == item_id]
    return [g for g in genre_columns if row[g].values[0] == 1]

In [81]:
for item in recs:
    print(item, get_genres(item))

121 ['Action', 'Sci-Fi', 'War']
258 ['Drama', 'Sci-Fi']
100 ['Crime', 'Drama', 'Thriller']
748 ['Action', 'Romance', 'Thriller']
313 ['Action', 'Drama', 'Romance']
678 ['Drama', 'Thriller']
125 ['Drama', 'Romance']
268 ['Drama', 'Romance']
15 ['Drama']
271 ['Action', 'Adventure', 'Sci-Fi', 'War']


## Genre Analysis

The recommended items span multiple genres, including Drama, Action, Sci-Fi, Romance, and Thriller. While Drama appears to be the dominant genre, the model does not collapse into a single category and instead provides a diverse set of recommendations.

This suggests that the model successfully captures user preferences while maintaining a balance between relevance and diversity. The inclusion of genre features allows the model to generalize beyond item IDs and recommend semantically related items across multiple categories.

In [82]:
user_id = best_users[0][0]

user_history = train_df[train_df["user_id"] == user_id]["item_id"].tolist()

show_recommendations_with_titles(user_history[:10])


475: Trainspotting (1996)
825: Arrival, The (1996)
741: Last Supper, The (1995)
127: Godfather, The (1972)
181: Return of the Jedi (1983)
117: Rock, The (1996)
280: Up Close and Personal (1996)
300: Air Force One (1997)
284: Tin Cup (1996)
282: Time to Kill, A (1996)


## Qualitative Recommendation Analysis

The recommended items include well-known and high-quality movies such as *The Godfather*, *Trainspotting*, *Return of the Jedi*, and *Air Force One*. These recommendations span multiple genres, including drama, crime, action, and sci-fi.

This suggests that the model successfully captures user preferences and retrieves relevant items from different categories. However, the recommendations also tend to favor relatively popular and mainstream movies, indicating a moderate popularity bias and a preference for safe recommendations over exploration.

In [83]:
from sklearn.metrics.pairwise import cosine_similarity
import itertools
import numpy as np

## Diversity Analysis

In [84]:
def diversity_of_recommendation_list(item_ids, item_features, genre_columns):
    # önerilen item'ların genre vektörlerini al
    feature_matrix = item_features.set_index("movie_id").loc[item_ids, genre_columns].values

    # eğer 2'den az item varsa diversity anlamsız olur
    if len(feature_matrix) < 2:
        return 0.0

    sims = []
    for i, j in itertools.combinations(range(len(feature_matrix)), 2):
        sim = cosine_similarity(
            feature_matrix[i].reshape(1, -1),
            feature_matrix[j].reshape(1, -1)
        )[0][0]
        sims.append(sim)

    avg_similarity = np.mean(sims)
    diversity = 1 - avg_similarity
    return diversity

In [85]:
user_id = 1
recs = recommend_faiss_filtered(user_id, top_k=10, search_k=100)

div_score = diversity_of_recommendation_list(recs, item_features, genre_columns)
print("Diversity score:", div_score)

Diversity score: 0.7292678411836517


In [89]:
from sklearn.metrics.pairwise import cosine_similarity
import itertools
import numpy as np

def diversity_of_recommendation_list(item_ids, item_features, genre_columns):
    feature_matrix = item_features.set_index("movie_id").loc[item_ids, genre_columns].values

    if len(feature_matrix) < 2:
        return 0.0

    sims = []
    for i, j in itertools.combinations(range(len(feature_matrix)), 2):
        sim = cosine_similarity(
            feature_matrix[i].reshape(1, -1),
            feature_matrix[j].reshape(1, -1)
        )[0][0]
        sims.append(sim)

    avg_similarity = np.mean(sims)
    diversity = 1 - avg_similarity
    return diversity

def average_diversity_faiss(test_df, top_k=10, search_k=100):
    diversities = []

    test_user_ids = test_df["user_id"].unique()

    for user_id in test_user_ids:
        if user_id not in user2idx:
            continue

        recs = recommend_faiss_filtered(user_id, top_k=top_k, search_k=search_k)

        if len(recs) >= 2:
            div_score = diversity_of_recommendation_list(recs, item_features, genre_columns)
            diversities.append(div_score)

    return np.mean(diversities)

avg_diversity = average_diversity_faiss(test_df, top_k=10, search_k=100)
print("Average Diversity:", avg_diversity)

Average Diversity: 0.6797934742458354


## Diversity Analysis

To evaluate recommendation variety, a diversity metric was computed based on genre-level cosine dissimilarity among the top-10 recommended items.

A higher diversity score indicates that the recommendation list contains items from more varied genres, while a lower score suggests that the system concentrates on highly similar items.

This metric complements accuracy-based metrics such as Recall@10 and Precision@10 by showing whether the system balances relevance with recommendation variety.

## Production Pipeline

To make the recommendation system suitable for real-world usage, the overall pipeline can be separated into **offline** and **online** stages. This design allows computationally expensive tasks to be performed in advance, while keeping inference efficient at serving time.

## Offline Stage

In the offline stage, the recommendation model is trained using historical user-item interaction data. During training, the Two-Tower architecture learns user and item representations, while item-side genre features enrich the item embeddings.

After the training process is completed, item embeddings are generated for all items in the catalog. These embeddings are then stored and indexed using FAISS, which enables efficient nearest-neighbor retrieval during inference.

The offline stage consists of the following steps:

1. Data preprocessing and interaction construction  
2. User and item indexing  
3. Negative sampling  
4. Model training  
5. Item embedding generation  
6. FAISS index creation

## Online Inference Stage

In the online stage, the system responds to a user request in real time. First, the trained user tower generates the embedding representation of the target user. This user embedding is then used to query the FAISS index in order to retrieve the most similar candidate items.

After retrieval, previously seen items are filtered out so that the user is not recommended items they have already interacted with. The remaining items are returned as the final recommendation list.

The online stage consists of the following steps:

1. Receive user request  
2. Compute user embedding  
3. Query FAISS index  
4. Retrieve top-K candidate items  
5. Filter out previously seen items  
6. Return personalized recommendations

## End-to-End System Flow

The overall recommendation system works as follows:

### Offline
- Train the Two-Tower recommendation model
- Generate item embeddings
- Build a FAISS index over item embeddings

### Online
- Generate a user embedding for the active user
- Search the FAISS index for the nearest items
- Remove already seen items
- Return the final recommendation list

This architecture enables scalable recommendation generation while maintaining strong retrieval quality.

## Why This Design Matters

Separating the system into offline and online stages is important for scalability and efficiency. Model training and embedding generation are computationally expensive, so they are handled offline. In contrast, real-time recommendation serving must be fast, which is why FAISS-based nearest-neighbor retrieval is used during inference.

This design makes the system more practical for large-scale recommendation scenarios, where scoring all items one by one would be too slow.

## Production Perspective

From a production perspective, this architecture provides several advantages:

- **Scalability:** item retrieval remains efficient even when the item catalog grows  
- **Efficiency:** expensive computations are moved offline  
- **Personalization:** recommendations are generated based on learned user preferences  
- **Flexibility:** the model can be improved further by adding more user or item features  
- **Extensibility:** a ranking stage can later be added after retrieval for even better recommendation quality

## Summary

The final system is not only a machine learning model, but an end-to-end recommendation pipeline that combines representation learning, feature engineering, efficient retrieval, and real-time serving logic.

## Limitations

Despite achieving strong performance, the current recommendation system has several limitations.

First, the model relies primarily on user-item interactions and item-side genre features. It does not incorporate additional user-specific features such as demographics or behavioral signals, which could further improve personalization.

Second, the system does not explicitly model temporal dynamics. User preferences may change over time, but the current approach treats all interactions equally regardless of when they occurred.

Third, although FAISS enables efficient retrieval, it introduces a slight trade-off in accuracy compared to full scoring. This means that some relevant items may not always be retrieved in the top-K candidates.

Fourth, the model shows a moderate popularity bias, tending to recommend relatively well-known items more frequently than niche or less popular ones.

Finally, the system operates as a single-stage retrieval model. While it performs well in candidate generation, it does not include a second-stage ranking model to refine and re-order the recommendations.

## Future Work

Several improvements can be made to further enhance the recommendation system.

A key extension would be to introduce a **ranking model** as a second stage. In this setup, FAISS would first retrieve a set of candidate items (e.g., top 100), and a more complex model would then rank these candidates to improve precision.

Incorporating additional features could also improve performance. For example, user-side features (such as interaction frequency or preferences) and richer item metadata (such as textual descriptions or embeddings from pretrained models) could be added.

Another improvement would be to introduce **temporal modeling**, allowing the system to capture how user preferences evolve over time.

To reduce popularity bias, techniques such as re-ranking or diversity-aware optimization could be applied to promote less popular but relevant items.

Finally, the system could be extended with **online learning** or **A/B testing** to continuously evaluate and improve recommendation performance in a real-world setting.